# Batch Scoring — Task Delay Prediction

Scores every row with the champion **registered model** from the MLflow registry.

**Reads:** `gold_ml_features`, `gold_ml_scoring_meta`, MLflow registry  **Writes:** `gold_ml_predictions`, `gold_ml_summary`

In [ ]:
import json
import mlflow
import mlflow.lightgbm
import mlflow.sklearn
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, current_timestamp, when, avg, count,
    sum as spark_sum, round as spark_round
)

spark = SparkSession.builder.getOrCreate()

# Load champion registered model + scoring metadata (flavor-aware: predict_proba)
meta = json.loads(spark.read.table('gold_ml_scoring_meta').collect()[0]['meta_json'])
champion_name = meta['champion_model']
champion_flavor = meta['champion_flavor']
feature_cols = meta['feature_cols']
numeric_features = meta['numeric_features']
cat_cols = meta['cat_cols']
category_maps = meta['category_maps']

uri = f'models:/{champion_name}/latest'
model = mlflow.lightgbm.load_model(uri) if champion_flavor == 'lightgbm' else mlflow.sklearn.load_model(uri)
df = spark.read.table('gold_ml_features')
print(f'Loaded registered model: {champion_name}')
print(f'Scoring {df.count():,} feature rows')

In [ ]:
# Prepare features exactly as in training: named float64 pandas columns
pdf = df.toPandas()
for c in numeric_features:
    pdf[c] = pd.to_numeric(pdf[c], errors='coerce').fillna(0.0)
for c in cat_cols:
    pdf[f'{c}_idx'] = pdf[c].astype(str).map(category_maps[c]).fillna(-1).astype('float64')

X_all = pdf[feature_cols].astype('float64')
pdf['probability_positive'] = model.predict_proba(X_all)[:, 1]
pdf['prediction'] = model.predict(X_all).astype('int64')
print(f'Scored {len(pdf):,} rows with {champion_name}')

In [ ]:
scored = spark.createDataFrame(pdf)

predictions = (
    scored
    .withColumn('delay_probability', spark_round(col('probability_positive'), 4))
    .withColumn('predicted_delay', col('prediction').cast('int'))
    .withColumn('risk_level',
        when(col('delay_probability') > 0.8, 'critical')
        .when(col('delay_probability') > 0.6, 'high')
        .when(col('delay_probability') > 0.4, 'medium')
        .otherwise('low'))
    .withColumn('scored_at', current_timestamp())
    .select(
        'task_id', 'project_id', 'assigned_subcontractor_id', 'sub_trade', 'task_name', 'project_type',
        'planned_duration_days', 'sub_rating',
        'had_delay', 'predicted_delay', 'delay_probability', 'risk_level',
        'scored_at')
)
predictions.write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable('gold_ml_predictions')
print(f'Predictions written: {predictions.count():,} rows')
predictions.groupBy('risk_level').count().orderBy('count', ascending=False).show()

In [ ]:
# Per-trade delay risk summary
summary = (
    predictions
    .groupBy('sub_trade')
    .agg(
        count('*').alias('total_tasks'),
        spark_sum('predicted_delay').alias('predicted_delay_count'),
        spark_sum('had_delay').alias('actual_delay_count'),
        spark_round(avg('delay_probability'), 4).alias('avg_delay_probability'),
        spark_round(avg('planned_duration_days'), 1).alias('avg_duration_days'),
        spark_round(avg('sub_rating'), 2).alias('avg_sub_rating'),
    )
    .withColumn('delay_rate', spark_round(col('predicted_delay_count') / col('total_tasks') * 100, 1))
    .withColumn('overall_risk',
        when(col('avg_delay_probability') > 0.6, 'high')
        .when(col('avg_delay_probability') > 0.3, 'medium')
        .otherwise('low'))
    .withColumn('summary_timestamp', current_timestamp())
    .withColumnRenamed('sub_trade', 'trade')
    .orderBy(col('avg_delay_probability').desc())
)
summary.write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable('gold_ml_summary')
print(f'Trade delay summary: {summary.count()} rows')
summary.show(15, truncate=False)

In [ ]:
spark.sql('OPTIMIZE gold_ml_predictions')
spark.sql('OPTIMIZE gold_ml_summary')
print('All Gold ML tables optimized')